# Week 4 &mdash; 3D Data and Point Clouds

## Implementation: PointNet from Scratch and the Rotation Problem

We implement a minimal **PointNet** in PyTorch, *empirically verify* its permutation invariance, *demonstrate* its lack of rotation invariance, and quantify the effect of rotation **augmentation**. The exercises point to fully equivariant alternatives.

### Setup


In [ ]:
# !pip install numpy torch matplotlib

import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

torch.manual_seed(0)
np.random.seed(0)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


## 1. A Minimal PointNet

Following the theory notebook, our classifier is
$$
f(\{p_i\}) = \gamma\!\Big(\max_i h(p_i)\Big),
$$
with $h$ and $\gamma$ implemented as MLPs and `max` over the point dimension as the symmetric pooling operator.


In [ ]:
class PointNet(nn.Module):
    def __init__(self, in_dim=3, emb=256, n_classes=4):
        super().__init__()
        # Shared per-point MLP h: applied identically to every point.
        self.h = nn.Sequential(
            nn.Linear(in_dim, 64), nn.ReLU(),
            nn.Linear(64, 128), nn.ReLU(),
            nn.Linear(128, emb), nn.ReLU(),
        )
        # Classification head gamma on the pooled global feature.
        self.gamma = nn.Sequential(
            nn.Linear(emb, 128), nn.ReLU(),
            nn.Linear(128, n_classes),
        )

    def forward(self, x):
        # x: (batch, N, 3)
        feats = self.h(x)                 # (batch, N, emb)
        global_feat, _ = feats.max(dim=1) # symmetric pooling -> (batch, emb)
        return self.gamma(global_feat)

model = PointNet().to(device)
print(model)


## 2. A Synthetic Point-Cloud Dataset

To keep the notebook self-contained we generate clouds sampled from four canonical shapes: **sphere**, **cube surface**, **plane**, and **cylinder**. Each is a permutation-invariant set of 3D points.


In [ ]:
def sample_sphere(n):
    v = np.random.randn(n, 3); return v / np.linalg.norm(v, axis=1, keepdims=True)

def sample_cube(n):
    p = np.random.uniform(-1, 1, (n, 3))
    face = np.random.randint(0, 3, n)
    sign = np.random.choice([-1, 1], n)
    p[np.arange(n), face] = sign
    return p

def sample_plane(n):
    p = np.random.uniform(-1, 1, (n, 3)); p[:, 2] = 0.0; return p

def sample_cylinder(n):
    t = np.random.uniform(0, 2*np.pi, n)
    z = np.random.uniform(-1, 1, n)
    return np.stack([np.cos(t), np.sin(t), z], axis=1)

GENERATORS = [sample_sphere, sample_cube, sample_plane, sample_cylinder]
NAMES = ["sphere", "cube", "plane", "cylinder"]

def make_batch(batch=64, n_pts=256):
    X, y = [], []
    for _ in range(batch):
        c = np.random.randint(4)
        X.append(GENERATORS[c](n_pts)); y.append(c)
    X = torch.tensor(np.array(X), dtype=torch.float32)
    y = torch.tensor(y, dtype=torch.long)
    return X.to(device), y.to(device)

# Quick visual sanity check.
fig = plt.figure(figsize=(11, 3))
for i, (g, name) in enumerate(zip(GENERATORS, NAMES)):
    pts = g(300)
    ax = fig.add_subplot(1, 4, i+1, projection="3d")
    ax.scatter(pts[:,0], pts[:,1], pts[:,2], s=4)
    ax.set_title(name); ax.set_box_aspect((1,1,1)); ax.axis("off")
plt.tight_layout(); plt.show()


## 3. Training PointNet


In [ ]:
opt = torch.optim.Adam(model.parameters(), lr=1e-3)
loss_fn = nn.CrossEntropyLoss()

model.train()
for step in range(1, 401):
    X, y = make_batch()
    opt.zero_grad()
    loss = loss_fn(model(X), y)
    loss.backward(); opt.step()
    if step % 100 == 0:
        with torch.no_grad():
            Xt, yt = make_batch(256)
            acc = (model(Xt).argmax(1) == yt).float().mean().item()
        print(f"step {step:3d} | loss {loss.item():.4f} | test acc {acc:.3f}")


## 4. Empirical Test 1 &mdash; Permutation Invariance

We feed the *same* cloud with its points shuffled and confirm the logits are (numerically) identical.


In [ ]:
model.eval()
X, _ = make_batch(1, 256)
with torch.no_grad():
    out_orig = model(X)
    perm = torch.randperm(X.shape[1])
    out_perm = model(X[:, perm, :])

print("Max |logit difference| under point permutation:",
      (out_orig - out_perm).abs().max().item())


The difference is at floating-point precision: **PointNet is exactly permutation-invariant**, exactly as the Deep Sets theorem predicts.


## 5. Empirical Test 2 &mdash; PointNet Is *Not* Rotation-Invariant

We apply random $SO(3)$ rotations to test clouds (which leaves their true class unchanged) and measure the drop in accuracy of the model trained *without* rotation augmentation.


In [ ]:
def random_rotation_matrix():
    # Uniform random rotation via QR of a Gaussian matrix.
    A = np.random.randn(3, 3)
    Q, R = np.linalg.qr(A)
    Q = Q @ np.diag(np.sign(np.diag(R)))
    if np.linalg.det(Q) < 0:
        Q[:, 0] *= -1
    return torch.tensor(Q, dtype=torch.float32, device=device)

def rotated_accuracy(model, n_batches=8):
    accs = []
    with torch.no_grad():
        for _ in range(n_batches):
            X, y = make_batch(128)
            Rm = random_rotation_matrix()
            Xr = X @ Rm.T                        # rotate every point
            accs.append((model(Xr).argmax(1) == y).float().mean().item())
    return float(np.mean(accs))

with torch.no_grad():
    Xt, yt = make_batch(1024)
    clean_acc = (model(Xt).argmax(1) == yt).float().mean().item()

print(f"Accuracy on un-rotated clouds : {clean_acc:.3f}")
print(f"Accuracy on randomly rotated  : {rotated_accuracy(model):.3f}")


Accuracy collapses under rotation. The model has learned features tied to the *canonical orientation* of the training data &mdash; it is permutation-invariant but **not** $SO(3)$-invariant, precisely the limitation discussed in the theory notebook.


## 6. Remedy &mdash; Rotation Augmentation

We retrain from scratch, applying a random rotation to *every* training cloud. This induces approximate, statistical rotation invariance.


In [ ]:
def make_batch_rotated(batch=64, n_pts=256):
    X, y = make_batch(batch, n_pts)
    out = []
    for i in range(X.shape[0]):
        out.append(X[i] @ random_rotation_matrix().T)
    return torch.stack(out), y

aug_model = PointNet().to(device)
opt = torch.optim.Adam(aug_model.parameters(), lr=1e-3)
aug_model.train()
for step in range(1, 401):
    X, y = make_batch_rotated()
    opt.zero_grad()
    loss = loss_fn(aug_model(X), y)
    loss.backward(); opt.step()

aug_model.eval()
print(f"Augmented model | clean acc   : {(aug_model(Xt).argmax(1)==yt).float().mean().item():.3f}")
print(f"Augmented model | rotated acc : {rotated_accuracy(aug_model):.3f}")


With augmentation, rotated-test accuracy recovers substantially. This is **approximate** invariance achieved through data, not architecture &mdash; in contrast to the *exact, built-in* equivariance of Tensor Field Networks. The trade-off (sample cost and no guarantees vs. architectural complexity) is a recurring theme in geometric deep learning.


## 7. Summary and Exercises

- We implemented **PointNet** = shared MLP + max-pool + head.
- We confirmed **permutation invariance** holds exactly and by construction.
- We demonstrated that **rotation invariance does not** hold for raw PointNet, and that **augmentation** recovers it approximately.

### Exercises

1. Replace `max` pooling with `mean` and `sum`. Measure classification accuracy and discuss expressivity (cf. Week 3 GIN).
2. Build a *rotation-invariant input*: feed the model the sorted vector of pairwise distances to the $k$ nearest neighbours instead of raw coordinates. Does rotated accuracy now match clean accuracy?
3. Install **`e3nn`** and implement a single $SO(3)$-equivariant convolution. Verify that a type-1 (vector) output rotates as $v \mapsto Rv$ when the input cloud is rotated by $R$.
